# Scrape full job details for each job

For each job (you can go by URL), please try to get the following pieces of information:
1. Job Description
2. Do they ask additional questions?
3. Company Information (May not exist for some firms)
• Company name

• Industry

• Firm size

• Company description

• Perks and benefits

• Average rating

• Number of reviews

• Any other information

Please get this information for the most recent 1000 jobs and report how long the
scraping takes

# Selenium

In [ ]:
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
import re

# Set up the driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

def get_job_urls_from_page(page_num):
    URL = f"https://sg.jobstreet.com/jobs?page={page_num}&sortmode=ListedDate"
    driver.get(URL)
    time.sleep(3)  # Wait for the page to load

    job_urls = []
    job_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="job-card"]')
    
    for job in job_cards:
        url_tag = job.find_element(By.CSS_SELECTOR, '[data-automation="job-list-view-job-link"]')
        job_url = url_tag.get_attribute('href') if url_tag else 'N/A'
        job_urls.append(job_url)
    
    return job_urls

# Get job URLs from multiple pages
job_urls = []
page_num = 1
while len(job_urls) < 10:
    job_urls.extend(get_job_urls_from_page(page_num))
    page_num += 1
    if len(job_urls) >= 10:
        break

# Remove duplicates (if any)
job_urls = list(set(job_urls))[:10]


In [8]:
len(job_urls)

10

In [40]:
def get_job_details(job_url):
    driver.get(job_url)
    time.sleep(3)  # Wait for the job page to load

    job_details = {}

    # Job Title (for clarity)
    job_title_tag = driver.find_element(By.CSS_SELECTOR, '[data-automation="job-detail-title"]')
    job_details['Job Title'] = job_title_tag.text.strip() if job_title_tag else 'N/A'

    # Advertiser Name (for clarity)
    company_title_tag = driver.find_element(By.CSS_SELECTOR, '[data-automation="advertiser-name"]')
    job_details['Advertiser Name'] = company_title_tag.text.strip() if company_title_tag else 'N/A'

    # Job URL to trace back
    job_details['Job URL'] = job_url

    # Job Description
    try:
        description_tag = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-automation="jobAdDetails"]'))
        )
        job_details['Job Description'] = description_tag.text.strip() if description_tag else 'N/A'
    except Exception as e:
        job_details['Job Description'] = 'N/A'

    # Employer Questions
    try:
        questions_section = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//h2[contains(text(), 'Employer questions')]/following-sibling::div//ul"))
        )
        question_items = questions_section.find_elements(By.XPATH, ".//li")
        questions = [item.text.strip() for item in question_items if item.text.strip()]
        job_details['Employer Questions'] = questions if questions else ['N/A']
    except Exception as e:
        job_details['Employer Questions'] = ['N/A']
    

    # Company Profile
    try:
        company_section = driver.find_element(By.CSS_SELECTOR, '[data-automation="company-profile"]')
        
        # Company Name
        try:
            company_name = company_section.find_element(By.CSS_SELECTOR, 'button h4')
            job_details["Company Name"] = company_name.text.strip()
        except Exception as e:
            job_details['Company Name'] = 'N/A'
        
        # Industry
        try:
            industry_tag = company_section.find_element(By.XPATH, "//span[@class='gepq850 eihuid5b']/span[@class='gepq850 eihuid4z eihuidr'][1]")
            job_details["Industry"] = industry_tag.text.strip()
        except Exception as e:
            job_details['Industry'] = 'N/A'
        
        # Firm Size
        try:
            firm_size = company_section.find_element(By.XPATH, "//span[contains(text(),'employees')]")
            firm_size_strip = firm_size.text.strip() if firm_size else 'N/A'
            firm_size_clean = firm_size_strip.replace(" employees", "")
            job_details["Firm Size (number of employees)"] = firm_size_clean
        except Exception as e:
            job_details['Firm Size (number of employees)'] = 'N/A'
        
        # Company Description
        try:
            description_elements = driver.find_elements(By.CSS_SELECTOR, 'p.gepq850.eihuidcz, p.gepq850.eihuidcb')
        
            company_description = ' '.join([desc.text.strip() for desc in description_elements])
            
            job_details["Company Description"] = company_description
        except Exception as e:
            job_details['Company Description'] = 'N/A'

        
        # Perks and Benefits
        try:
            perks_elements = driver.find_elements(By.CSS_SELECTOR, 'div.gepq850._2l3v4k0')
            perks = [perk.text.strip() for perk in perks_elements if perk.text.strip()] if perks_elements else 'N/A'
            job_details["Perks and Benefits"] = perks
        except Exception as e:
            job_details['Perks and Benefits'] = 'N/A'
        
        # Average Rating
        try:
            avg_rating = company_section.find_element(By.CSS_SELECTOR, '[data-automation="company-profile-review-rating"]')
            job_details["Average Rating (out of 5)"] = avg_rating.text.strip() if avg_rating else 'N/A'
        except Exception as e:
            job_details['Average Rating (out of 5)'] = 'N/A'
        
        # Number of Reviews
        try:
            num_reviews = company_section.find_element(By.CSS_SELECTOR, '[data-automation="company-profile-review-link"]')
            num_reviews_strip = num_reviews.text.strip() if num_reviews else 'N/A'
            number_of_reviews_clean = int(re.search(r'\d+', num_reviews_strip).group(0))
            
            job_details["Number of Reviews"] = number_of_reviews_clean
        except Exception as e:
            job_details['Number of Reviews'] = 'N/A'
        
    except Exception as e:
        job_details['Company Name'] = 'N/A'
        job_details['Industry'] = 'N/A'
        job_details['Firm Size (number of employees)'] = 'N/A'
        job_details['Company Description'] = 'N/A'
        job_details['Perks and Benefits'] = 'N/A'
        job_details['Average Rating (out of 5)'] = 'N/A'
        job_details['Number of Reviews'] = 'N/A'

    return job_details


# Extract details for each job URL
job_data = []
for job_url in job_urls:
    job_details = get_job_details(job_url)
    job_data.append(job_details)

# Create DataFrame
df = pd.DataFrame(job_data)


In [41]:
df.head(10)

,Job Title,Advertiser Name,Job URL,Job Description,Employer Questions,Company Name,Industry,Firm Size (number of employees),Company Description,Perks and Benefits,Average Rating (out of 5),Number of Reviews
0,"Boutique Supervisor – $3,500 (ID: 668691)",PERSOLKELLY Singapore Pte Ltd (Formerly Kelly ...,https://sg.jobstreet.com/job/83144353?type=sta...,Responsibilities: \nLead and supervise retail ...,[Which of the following statements best descri...,Persolkelly,Human Resources & Recruitment,51-100,PERSOLKELLY is one of the largest recru...,N/A,2.7,7
1,"Lead Research Engineer, Industry Development (...","Agency for Science, Technology and Research (A...",https://sg.jobstreet.com/job/83142265?type=sta...,"Candidate will be required to develop, secure ...",[N/A],"Agency for Science, Technology and Research",Government & Defence,"101-1,000","About the Agency for Science, Technology and...",N/A,3.8,23
2,Sales Admin,Ensign InfoSecurity (Singapore) Pte. Ltd.,https://sg.jobstreet.com/job/83144151?type=sta...,Key Responsibilities:\nService Request Managem...,[Which of the following statements best descri...,Ensign Infosecurity,Computer Software & Networking,"101-1,000",Ensign InfoSecurity is the largest pure-play ...,N/A,4.3,3
3,Editor,UOB Kay Hian Pte Ltd,https://sg.jobstreet.com/job/83142614?type=sta...,Are you a detail-oriented professional with a ...,[Which of the following statements best descri...,UOB Kay Hian,Banking & Financial Services,"101-1,000","UOB Kay Hian Pte Ltd, the largest stockbrokin...","[Medical, Dental]",4.1,15
4,Sales Manager (Japanese Speaking),Pasona Singapore Pte. Ltd.,https://sg.jobstreet.com/job/83143969?type=sta...,Location: Jurong East\nPosition: Sales Manager...,[Which of the following statements best descri...,Pasona,Accounting,11-50,Pasona Singapore Pte. Ltd. is a Japanese r...,"[Hybrid Working Environment, Employee's Well-b...",4.2,6
5,Financial Controller,Private Advertiser,https://sg.jobstreet.com/job/83142189?type=sta...,About the role\nWe are seeking a Financial Con...,[Which of the following statements best descri...,N/A,N/A,N/A,N/A,N/A,N/A,N/A
6,Assistant Reservations Manager,Pasona Singapore Pte. Ltd.,https://sg.jobstreet.com/job/83144180?type=sta...,Location: Central\nPosition: Assistant Reserva...,[Which of the following statements best descri...,Pasona,Accounting,11-50,Pasona Singapore Pte. Ltd. is a Japanese r...,"[Hybrid Working Environment, Employee's Well-b...",4.2,6
7,Admin Executive,Enova Electrical Pte Ltd,https://sg.jobstreet.com/job/83143901?type=sta...,Job Duties:\nProvide support for projects via ...,[Which of the following types of qualification...,Enova Electrical Pte Ltd,Consumer Electronics Manufacturing,11-50,Enova Electrical Pte Ltd is a premium switc...,[Medical],N/A,N/A
8,Accounts Executive,Enova Electrical Pte Ltd,https://sg.jobstreet.com/job/83143960?type=sta...,RESPONSIBILITIES:\nResponsible for full set of...,[Which of the following statements best descri...,Enova Electrical Pte Ltd,Consumer Electronics Manufacturing,11-50,Enova Electrical Pte Ltd is a premium switc...,[Medical],N/A,N/A
9,"Senior/Lead Research Engineer, Process Integra...","Agency for Science, Technology and Research (A...",https://sg.jobstreet.com/job/83144377?type=sta...,Job Description:\nResponsible for proposing an...,[N/A],"Agency for Science, Technology and Research",Government & Defence,"101-1,000","About the Agency for Science, Technology and...",N/A,3.8,23


In [ ]:
# Save to Excel
df.to_excel("10_job_details_with_questions.xlsx", index=False)
print("Job details saved to Excel file.")

In [42]:
driver.quit()

37m 10.2s for 200 jobs with job description and additional questions only

3m 26s for 20 jobs with job description and additional questions only

6m 51s for 20 jobs with all job details (with errors)

2m for 10 jobs with all job details (correct)